# 03 — Modelo de ML: Detecção de Fraude

**Objetivo:** treinar e avaliar um classificador de fraudes usando a base preparada no notebook 02, comparar com o baseline (`fraud_score`) e inspecionar o que o modelo aprendeu.

| Passo | O que faz |
|---|---|
| 1 | Carrega treino e teste já prontos de `data/processed/` |
| 2 | Pré-processa: imputa nulos, codifica categorias (Pipeline sklearn) |
| 3 | Treina **Random Forest** (simples, interpretável, bom em dados mistos) |
| 4 | Avalia no **teste com proporção real** — PR-AUC, Recall, Precision, F1 |
| 5 | Compara com o baseline `fraud_score` |
| 6 | Escolhe um limiar de decisão pelo custo (fraude perdida × falso alarme) |
| 7 | Inspeciona importância das features e dois exemplos de predição |

> **Por que Random Forest?** Lida bem com categorias, nulos imputados, escalas diferentes e dados desbalanceados. É fácil de explicar para o agente. Se o objetivo fosse máxima performance, usaríamos XGBoost — fica como próximo passo.

## 1. Carga

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC = ROOT / "data" / "processed"

SEMENTE = 42
COR_LEGIT, COR_FRAUDE = "#2a78d6", "#eb6834"
sns.set_theme(style="whitegrid", rc={"axes.spines.top": False, "axes.spines.right": False})

treino = pd.read_csv(PROC / "treino_balanceado.csv")
teste  = pd.read_csv(PROC / "teste_proporcao_real.csv")

print(f"Treino : {len(treino):,} linhas | fraudes: {treino.feedback_label.sum()} ({treino.feedback_label.mean():.1%})")
print(f"Teste  : {len(teste):,}  linhas | fraudes: {teste.feedback_label.sum()}  ({teste.feedback_label.mean():.2%})")

## 2. Definição de features e pré-processamento

In [ ]:
# ── Features do modelo (sem IDs, sem alvo, sem scores externos) ──────────────
FEATURES_NUM = [
    "amount", "amount_fees",
    "acceptor_risk",
    "customer_presumed_income", "customer_reported_income",
    "account_available_credit", "account_balance", "card_credit_limit",
    "cardid_count_1m", "cp_card_term_vel_1h",
    "cp_card_prev_term_dist_1h", "digitalwallettoken_count_3m",
    "hora", "dia_semana",
    "valor_vs_media_cartao", "valor_vs_limite",
]
FEATURES_CAT = [
    "tipo_transacao", "modalidade_transacao", "card_presence_flag",
    "card_type", "card_brand", "mcc",
    "acceptor_region",
    "customer_risk_rating", "customer_region",
]
FEATURES_BIN = ["madrugada", "cnp_credito", "primeira_compra_cartao"]
SCORES_EXTERNOS = ["fraud_score", "visa_fraud_score", "mc_fraud_score"]
ALVO = "feedback_label"

# Mantém só as que existem nas bases (robustez se alguma coluna ainda não estiver lá)
cols_disponiveis = set(treino.columns)
FEATURES_NUM = [c for c in FEATURES_NUM if c in cols_disponiveis]
FEATURES_CAT = [c for c in FEATURES_CAT if c in cols_disponiveis]
FEATURES_BIN = [c for c in FEATURES_BIN if c in cols_disponiveis]
TODAS_FEATURES = FEATURES_NUM + FEATURES_CAT + FEATURES_BIN

X_treino = treino[TODAS_FEATURES]
y_treino = treino[ALVO].astype(int)
X_teste  = teste[TODAS_FEATURES]
y_teste  = teste[ALVO].astype(int)

# ── Pipeline de pré-processamento ────────────────────────────────────────────
# Numéricas: imputa com a mediana (robusto a outliers)
pipe_num = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
# Categóricas: converte para str primeiro (colunas com nulos lidos do CSV ficam como
# float+str mistos), imputa 'desconhecido' e codifica como inteiro ordinal.
from sklearn.preprocessing import FunctionTransformer
to_str = FunctionTransformer(lambda X: X.astype(str))
pipe_cat = Pipeline([
    ("to_str",  to_str),
    ("imputer", SimpleImputer(strategy="constant", fill_value="desconhecido")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
# Binárias: imputa com 0 (ausência do sinal)
pipe_bin = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
])

preprocessor = ColumnTransformer([
    ("num", pipe_num, FEATURES_NUM),
    ("cat", pipe_cat, FEATURES_CAT),
    ("bin", pipe_bin, FEATURES_BIN),
], remainder="drop")

print(f"Features: {len(TODAS_FEATURES)} ({len(FEATURES_NUM)} numéricas, "
      f"{len(FEATURES_CAT)} categóricas, {len(FEATURES_BIN)} binárias)")

## 3. Treinamento

In [ ]:
modelo = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,          # evita overfitting no treino pequeno
        min_samples_leaf=2,
        class_weight="balanced",  # reforça a classe minoritária (precaução extra)
        random_state=SEMENTE,
        n_jobs=-1,
    )),
])

modelo.fit(X_treino, y_treino)
print("Modelo treinado.")

## 4. Avaliação no teste com proporção real

> **Por que PR-AUC e não ROC-AUC?**  
> Com 0,09% de fraudes no teste, um modelo que acerta 99,9% das legítimas pode ter ROC-AUC alto mesmo sem detectar nada. PR-AUC (Precision-Recall) penaliza essa falsa segurança.

In [ ]:
y_proba = modelo.predict_proba(X_teste)[:, 1]

pr_auc  = average_precision_score(y_teste, y_proba)
roc_auc = roc_auc_score(y_teste, y_proba)
print(f"PR-AUC  (nosso modelo) : {pr_auc:.3f}")
print(f"ROC-AUC (nosso modelo) : {roc_auc:.3f}")

# ── Curva Precision-Recall ────────────────────────────────────────────────────
precision, recall, thresholds = precision_recall_curve(y_teste, y_proba)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(recall, precision, color=COR_FRAUDE, lw=2, label=f"Nosso modelo (PR-AUC = {pr_auc:.3f})")
ax.axhline(y_teste.mean(), color="gray", ls="--", label=f"Baseline aleatório ({y_teste.mean():.3f})")
ax.set_xlabel("Recall (% de fraudes detectadas)")
ax.set_ylabel("Precisão (% de alertas corretos)")
ax.set_title("Curva Precision-Recall — teste com proporção real")
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / "docs" / "assets" / "pr_curve.png", dpi=120)
plt.show()

## 5. Comparação com o baseline (`fraud_score`)

In [ ]:
scores_disponiveis = [c for c in SCORES_EXTERNOS if c in teste.columns]

resultados = {"Nosso modelo": {"PR-AUC": pr_auc, "ROC-AUC": roc_auc}}
for col in scores_disponiveis:
    s = teste[col].fillna(0)
    resultados[col] = {
        "PR-AUC":  average_precision_score(y_teste, s),
        "ROC-AUC": roc_auc_score(y_teste, s),
    }

df_res = pd.DataFrame(resultados).T.sort_values("PR-AUC", ascending=False)
display(df_res.style.format("{:.3f}").highlight_max(color="#d4edda"))

## 6. Escolha do limiar de decisão

Num problema de fraude o custo de **deixar passar uma fraude** é muito maior do que o custo de **incomodar um associado com um falso alarme**. Aqui usamos a heurística de F-beta com β=2 (recall vale o dobro da precisão). O agente pode ajustar esse limiar conforme a política do Sicredi.

In [ ]:
# Calcula F-beta para cada limiar (beta=2 → recall pesa 2x mais que precisão)
BETA = 2
fbeta = (1 + BETA**2) * (precision * recall) / (BETA**2 * precision + recall + 1e-9)
idx_melhor = np.argmax(fbeta[:-1])  # último elemento não tem threshold
limiar_ótimo = thresholds[idx_melhor]

print(f"Limiar ótimo (F{BETA})  : {limiar_ótimo:.3f}")
print(f"Precisão nesse limiar : {precision[idx_melhor]:.3f}")
print(f"Recall  nesse limiar  : {recall[idx_melhor]:.3f}")

# ── Relatório com o limiar escolhido ─────────────────────────────────────────
y_pred = (y_proba >= limiar_ótimo).astype(int)
print("\n" + classification_report(y_teste, y_pred, target_names=["legítima", "fraude"]))

# ── Visualização: limiar na curva PR ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(recall, precision, color=COR_FRAUDE, lw=2, label=f"PR-AUC = {pr_auc:.3f}")
ax.scatter(recall[idx_melhor], precision[idx_melhor],
           s=120, zorder=5, color="black",
           label=f"Limiar ótimo F{BETA} = {limiar_ótimo:.2f}\n"
                 f"(Recall {recall[idx_melhor]:.2f}, Prec {precision[idx_melhor]:.2f})")
ax.axhline(y_teste.mean(), color="gray", ls="--")
ax.set_xlabel("Recall")
ax.set_ylabel("Precisão")
ax.set_title("Limiar de decisão na curva PR")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 7. Importância das features

In [ ]:
# Recupera os nomes das colunas após o ColumnTransformer
nomes_finais = (
    FEATURES_NUM
    + FEATURES_CAT
    + FEATURES_BIN
)

importancias = modelo.named_steps["clf"].feature_importances_
df_imp = (
    pd.DataFrame({"feature": nomes_finais, "importancia": importancias})
    .sort_values("importancia", ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(8, 6))
cores = [COR_FRAUDE if f in FEATURES_BIN else COR_LEGIT for f in df_imp.feature]
ax.barh(df_imp.feature[::-1], df_imp.importancia[::-1], color=cores[::-1])
ax.set_xlabel("Importância (Gini)")
ax.set_title("Top 20 features — Random Forest")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color=COR_LEGIT,  label="original / pré-calculada"),
    Patch(color=COR_FRAUDE, label="criada no notebook 02"),
], fontsize=9)
plt.tight_layout()
plt.savefig(ROOT / "docs" / "assets" / "feature_importance.png", dpi=120)
plt.show()

## 8. Exemplos de predição explicada

Simula o que o agente receberia: uma transação com score alto e uma com score baixo, junto com os valores das features mais importantes para justificar o alerta.

In [ ]:
def explicar_transacao(row: pd.Series, score: float, limiar: float) -> None:
    """Imprime um resumo legível de uma predição — base para o agente explicar."""
    alerta = score >= limiar
    simbolo = "🚨 ALERTA" if alerta else "✅ OK"
    print(f"\n{'='*55}")
    print(f"  {simbolo} — Score de fraude: {score:.3f} (limiar: {limiar:.3f})")
    print(f"{'='*55}")

    campos = [
        ("Canal",             row.get("tipo_transacao", "—")),
        ("Modalidade",        row.get("modalidade_transacao", "—")),
        ("Cartão presente",   row.get("card_presence_flag", "—")),
        ("Valor",             f"R$ {row.get('amount', 0):,.2f}"),
        ("Valor/média cartão",f"{row.get('valor_vs_media_cartao', float('nan')):.1f}×" if not pd.isna(row.get('valor_vs_media_cartao')) else "1ª compra"),
        ("Hora",              f"{int(row.get('hora', -1))}h"),
        ("Madrugada",         "Sim" if row.get("madrugada") == 1 else "Não"),
        ("Risco estab.",      f"{row.get('acceptor_risk', float('nan')):.2f}" if not pd.isna(row.get('acceptor_risk')) else "—"),
        ("1ª compra cartão",  "Sim" if row.get("primeira_compra_cartao") == 1 else "Não"),
    ]
    for nome, valor in campos:
        print(f"  {nome:<22}: {valor}")

# Pega a fraude com maior score e a legítima com menor score no teste
teste_com_score = teste.copy()
teste_com_score["score_modelo"] = y_proba

fraude_exemplo   = teste_com_score[y_teste == 1].nlargest(1, "score_modelo").iloc[0]
legitima_exemplo = teste_com_score[y_teste == 0].nsmallest(1, "score_modelo").iloc[0]

explicar_transacao(fraude_exemplo,   fraude_exemplo["score_modelo"],   limiar_ótimo)
explicar_transacao(legitima_exemplo, legitima_exemplo["score_modelo"], limiar_ótimo)

## 9. Análise crítica do modelo

### Resultados obtidos

O modelo atingiu **PR-AUC = 1.000 e ROC-AUC = 1.000** no teste com proporção real — detectando as 29 fraudes sem nenhum falso alarme. Isso é tecnicamente perfeito, mas é exatamente aí que mora o problema.

### Por que PR-AUC = 1.0 é um sinal de alerta

Em produção real, fraudes e compras legítimas sempre se sobrepõem. Uma separação perfeita indica que o modelo encontrou um atalho — e encontrou: as **4 features de velocidade pré-calculadas** (`cp_card_term_vel_1h`, `digitalwallettoken_count_3m`, `cardid_count_1m`, `cp_card_prev_term_dist_1h`) respondem por **69% da importância total**.

Essas features já vieram prontas no dataset. Na base sintética, o gerador provavelmente as construiu com acesso à label, ou as exagerou propositalmente para facilitar o exercício. O efeito prático é que elas **denunciam a fraude diretamente**, e qualquer modelo que as use vai parecer perfeito — inclusive o baseline `fraud_score` (PR-AUC = 1.000).

### O que as features construídas pela equipe valem?

As 7 features criadas no notebook 02 (`madrugada`, `cnp_credito`, `valor_vs_media_cartao`, `valor_vs_limite`, `primeira_compra_cartao`, `hora`, `dia_semana`) somam apenas **~11% da importância** no modelo atual — não porque sejam fracas, mas porque as features de velocidade dominam antes delas ter chance.

Para avaliar o real poder preditivo dessas features, o próximo passo é rodar o modelo **sem as 4 pré-calculadas** e comparar os PR-AUCs.

### Implicação para a proposta agêntica

O resultado reforça o argumento central da EDA: **detectar a fraude nesta base já é um problema quase resolvido**. O diferencial da solução não está no classificador em si, mas no que acontece depois do alerta — explicar o motivo ao associado, pedir confirmação e tomar a decisão de bloquear ou liberar. É aí que a IA agêntica agrega valor real.